# Exploring decisions
It will help better understand those PDFs and do better chunking and single RAG

## Imports and paths

In [3]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pdfplumber # Great for inspecting PDF layout
from pathlib import Path
from tqdm.notebook import tqdm
from unidecode import unidecode
import re
import random
random.seed(42)
from collections import Counter


# style for charts
sns.set_theme(style="whitegrid")

# define project paths based on your structure
BASE_DIR = Path('..') 
RAW_DATA_DIR = BASE_DIR / 'data' / '01_raw_pdfs'

EDA_DIR = BASE_DIR / "data" / "99_eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW_DATA_DIR.resolve())
print("EDA:", EDA_DIR.resolve())

RAW: /Users/stefanec/STU_FIIT/bachelor-thesis-legal-text-information-extraction/data/01_raw_pdfs
EDA: /Users/stefanec/STU_FIIT/bachelor-thesis-legal-text-information-extraction/data/99_eda


## Definition - Text normalization + regex bank

In [23]:
def norm_pair(raw: str):
    """
    It prepares text for analysis - 2 versions
    Returns:
      raw_lower: lowercased original (keeps '§')
      ascii_norm: lower + unidecode (diacritics removed; '§' -> 'ss')
  I am doing it because of possibility of errors in PDFs
    """
    raw = raw or ""
    raw_lower = raw.lower()
    ascii_norm = unidecode(raw_lower)
    ascii_norm = re.sub(r"[ \t]+", " ", ascii_norm) # clean extra spaces and tabulators on one space
    return raw_lower, ascii_norm

def spaced_word_regex(word: str) -> str:
    # matches spaced-out headers like "o d o v o d n e n i e"
    return r"\s*".join(map(re.escape, word))

RE_ODOV = re.compile(r"odôvodnenie|odovodnenie", re.IGNORECASE)
RE_POUC = re.compile(r"poučenie|poucenie", re.IGNORECASE)
RE_VYROK = re.compile(r"(?:takto\s*)?rozhodol\s*:", re.IGNORECASE | re.DOTALL) # the previous regex missed verdict becsue it was split accross lines (now it allows newlines (\s*) between words "takto rozhodol"

# IMPORTANT: match both '§' (raw) and 'ss' (unidecode of '§')
PAR = r"(?:§|ss|s|par)\s*"

KEY = {
    # core
    "has_penalty": re.compile(r"\bzmluvn\w*\s+pokut\w*\b"),
    "has_301":     re.compile(
        rf"(?:{PAR}301\b)|(?:\b301\b.{0,80}\b(obchodn|obchz|obchodn\w*\s+zakonnik))", re.IGNORECASE),

    # civil noise
    "has_oz_545a": re.compile(PAR + r"545\s*a\b|" + PAR + r"545a\b"),
    "has_oz_544":  re.compile(PAR + r"544\b"),

    # moderation / proportionality signals
    "has_moder_trig": re.compile(r"\b(primeran\w*|neprimeran\w*|moder\w*|zniz\w*|poniz\w*|neprizna\w*|zamiet\w*)\b", re.IGNORECASE),

    # other grounds (to separate from moderation)
    "has_invalidity": re.compile(r"\b(neplatn\w*|neurcit\w*|dobr\w*\s+mrav\w*|poctiv\w*\s+obchodn\w*\s+styk\w*)\b", re.IGNORECASE),

    # interest confusion (rate != penalty)
    "has_interest": re.compile(r"\b(urok\w*|urok\s+z\s+omeskania|omeskan\w*)\b"),

    # candidates
    "has_percent": re.compile(r"(\d+(?:[.,]\d+)?)\s*%"),
    "has_money":   re.compile(r"(\d{1,3}(?:[ \.\u00A0]\d{3})*(?:,\d+)?|\d+(?:,\d+)?)\s*(eur|€|sk|sk\.)"),
    
    # civil noise filter
    "has_civil_code": re.compile(r"obciansk\w*\s+zakonnik", re.IGNORECASE),
}


## Load file registry

In [7]:
records = []

# structure: 01_raw_pdfs / {case_folder} / {file}.pdf
pdf_files = list(RAW_DATA_DIR.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files. Processing metadata...")

for pdf_path in tqdm(pdf_files):
    folder = pdf_path.parent
    
    # JSON: prefer exact match (file.pdf -> file.json)
    meta_path = pdf_path.with_suffix(".json")
    
    # Ak neexistuje presný názov, skúsime nájsť akýkoľvek JSON v priečinku (fallback)
    if meta_path.exists():
        json_files = [meta_path]
    else:
        json_files = list(folder.glob("*.json"))
    
    metadata = {}
    if json_files:
        try:
            with open(json_files[0], 'r', encoding='utf-8') as f:
                metadata = json.load(f)
        except Exception as e:
            print(f"Error reading JSON for {pdf_path.name}: {e}")

    # --- EXTRACTING FIELDS ---
    
    # 1. Court Name Logic (FIXED)
    # first checking _csv_metadata - there is actual court which is reasoning the decision
    csv_meta = metadata.get('_csv_metadata', {})
    court_name = csv_meta.get('court') 
    
    if not court_name:
        court_info = metadata.get('sud', {})
        court_name = court_info.get('nazov', 'Unknown')
    
    # 2. Date
    raw_date = metadata.get('datumVydania', None)
    
    # 3. Case ID
    case_id = metadata.get('spisovaZnacka', folder.name)
    
    # 4. Decision Type
    decision_form = metadata.get('formaRozhodnutia', 'Unknown')
    if isinstance(decision_form, list):
        decision_form = decision_form[0] if decision_form else "Unknown"
        
    # 5. Oblast (Subject) ### NEW - Dôležité pre filtrovanie!
    subject_list = metadata.get('oblast', [])
    # Ak je to list, spojíme ho do stringu, napr. "Obchodné právo"
    subject = ", ".join(subject_list) if isinstance(subject_list, list) else str(subject_list)
    
    # 6. ECLI
    ecli = metadata.get('ecli', 'N/A')

    # Creating record
    record = {
        'filename': pdf_path.name,
        'rel_path': str(pdf_path.relative_to(BASE_DIR)), # Relative to project root
        'file_size_kb': round(pdf_path.stat().st_size / 1024, 2),
        'court': court_name, 
        'date_str': raw_date,
        'case_id': case_id,
        'type': decision_form,
        'subject': subject, ### NEW
        'ecli': ecli,
        'raw_metadata': metadata 
    }
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Convert date column
df['date'] = pd.to_datetime(df['date_str'], format='%d.%m.%Y', errors='coerce', dayfirst=True)

# Zoradíme stĺpce pre krajší výpis
print(f"Loaded dataframe with {len(df)} rows.")
display(df[['case_id', 'court', 'subject', 'date', 'type']].head())

Found 176 PDF files. Processing metadata...


  0%|          | 0/176 [00:00<?, ?it/s]

Loaded dataframe with 176 rows.


,case_id,court,subject,date,type
0,35Cbi/59/2006,KS Banská Bystrica,Obchodné právo,2012-06-04,Rozsudok
1,B5-1CbZm/453/2014,KS Bratislava,Obchodné právo,2018-11-29,Rozsudok
2,6Co/65/2013,KS Trenčín,,2013-02-26,Uznesenie
3,31Cob/64/2011,KS Trnava,Obchodné právo,2011-12-20,Rozsudok
4,PN-10Cb/39/2009,KS Trnava,Obchodné právo,2011-10-25,Rozsudok


## Commercial scope report

In [11]:
# --- 1.X Smart Commercial Scope Filter ---
# Goal: Ensure we only keep Commercial Law documents.
# We check two things: 
# 1. Metadata from the court (often contains errors).
# 2. Case ID (Spisova znacka) - this is reliable (e.g., 'Cob' is always commercial).

# 1. Get the main category from metadata
df["meta_oblast"] = df["raw_metadata"].apply(
    lambda m: (m.get("oblast") or ["Unknown"])[0] if isinstance(m, dict) else "Unknown"
)

# 2. Function to check if Case ID belongs to Commercial Registry
# Markers: 'cb', 'cob', 'cozm' (bills of exchange), 'cbi', 'cbs'
def is_trade_register(case_id):
    cid = str(case_id).lower()
    trade_markers = ['cb', 'cob', 'cozm', 'cbi', 'cbs'] 
    return any(marker in cid for marker in trade_markers)

df["is_trade_reg"] = df["case_id"].apply(is_trade_register)

# 3. Assign a Status to each document for clarity
def get_status(row):
    if row["meta_oblast"] == "Obchodné právo":
        return "VALID (Metadata OK)"
    elif row["is_trade_reg"] == True:
        return "RESCUED (Bad metadata, but Trade ID)"
    else:
        return "DROP (Civil/Other)"

df["filter_status"] = df.apply(get_status, axis=1)

# --- REPORTING ---

print("=== DATASET CLEANING REPORT ===")
print("Distribution of documents by status:")
print(df["filter_status"].value_counts())
print("-" * 40)

# Show what we are dropping (to be sure)
dropped_df = df[df["filter_status"] == "DROP (Civil/Other)"]
if not dropped_df.empty:
    print(f"Examples of removed documents ({len(dropped_df)} total):")
    # Displaying clean table without index for better readability
    display(dropped_df[["case_id", "court", "meta_oblast"]].head(5))
else:
    print("No documents to drop. Dataset is clean.")

# --- FILTERING ---
# We keep everything that is NOT marked as DROP
original_count = len(df)
df = df[df["filter_status"] != "DROP (Civil/Other)"].copy()
final_count = len(df)

print("=" * 40)
print(f"SUMMARY:")
print(f"Original files: {original_count}")
print(f"Removed files:  {original_count - final_count}")
print(f"Final dataset:  {final_count}")
print("=" * 40)

=== DATASET CLEANING REPORT ===
Distribution of documents by status:
filter_status
VALID (Metadata OK)                     142
RESCUED (Bad metadata, but Trade ID)     10
Name: count, dtype: int64
----------------------------------------
No documents to drop. Dataset is clean.
SUMMARY:
Original files: 152
Removed files:  0
Final dataset:  152


## Target courts report

In [5]:
# 1.X Target courts report (KS + NS)
def court_tier(name: str) -> str:
    n = (name or "").lower()
    if "najvyšší súd" in n:
        return "NS"
    if "krajský súd" in n:
        return "KS"
    if "okresný súd" in n:
        return "OS"
    if "mestský súd" in n:
        return "MS"
    return "OTHER"

df["court_tier"] = df["court"].apply(court_tier)
print(df["court_tier"].value_counts())
print("KS+NS ratio:", (df["court_tier"].isin(["KS","NS"])).mean())


court_tier
KS    113
MS     30
OS     23
NS     10
Name: count, dtype: int64
KS+NS ratio: 0.6988636363636364


## PDF quality - smoke test

In [13]:
# --- 2.0 DETAILED PDF HEALTH CHECK ---
# Goal: Perform a deep audit of the PDF files to ensure text quality.
# We run 3 specific tests on every document sample:
# Test A: File Integrity (Can it be opened? Does it have pages?)
# Test B: Content Existence (Is there text, or is it a scanned image?)
# Test C: Encoding Quality (Is the text readable, or is it garbage symbols like '@#%')

def audit_pdf_health(pdf_path):
    """
    Analyzes the PDF and returns a detailed dictionary of metrics.
    We check the first 3 pages to get a representative sample.
    """
    stats = {
        "num_pages": 0,
        "char_count": 0,
        "valid_ratio": 0.0,  # Percentage of alphanumeric chars (a-z, 0-9)
        "status": "UNKNOWN",
        "snippet": ""
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # METRIC 1: Number of pages
            stats["num_pages"] = len(pdf.pages)
            
            if stats["num_pages"] == 0:
                stats["status"] = "FAILED: Empty File"
                return stats
            
            # Extract text from first 3 pages (or less if file is short)
            # We use 3 pages to skip potential cover sheets
            full_text = ""
            pages_to_check = min(3, stats["num_pages"])
            
            for i in range(pages_to_check):
                page_text = pdf.pages[i].extract_text()
                if page_text:
                    full_text += page_text + " "
            
            # METRIC 2: Total characters extracted
            stats["char_count"] = len(full_text)
            
            # TEST B: Content Existence
            if stats["char_count"] < 50: # Arbitrary threshold for "empty"
                stats["status"] = "FAILED: No Text (Scanned?)"
                return stats
            
            # METRIC 3: Valid Character Ratio (The "Garbage" Detector)
            # We count how many characters are letters or numbers.
            # If we extract "(CID:932) 0021", the ratio will be very low.
            valid_chars = sum(c.isalnum() for c in full_text)
            stats["valid_ratio"] = round(valid_chars / stats["char_count"], 3)
            
            stats["snippet"] = full_text[:100].replace('\n', ' ')

            # TEST C: Encoding Quality
            # If less than 40% of the text is alphanumeric, it is likely corrupted.
            if stats["valid_ratio"] < 0.40:
                stats["status"] = "FAILED: Garbage Encoding"
            else:
                stats["status"] = "PASSED (Healthy)"
                
            return stats

    except Exception as e:
        stats["status"] = f"CRITICAL ERROR: {str(e)}"
        return stats

# --- RUNNING THE AUDIT ---
print("Running detailed health check on sample documents...")

audit_data = []
# Taking a sample of 15 documents to be sure
sample_docs = df.head(15).copy()

for _, row in tqdm(sample_docs.iterrows(), total=len(sample_docs)):
    full_path = BASE_DIR / row['rel_path']
    
    # Run the audit function
    result = audit_pdf_health(full_path)
    
    # Add metadata for context
    result['filename'] = row['filename']
    result['case_id'] = row['case_id']
    audit_data.append(result)

# Create a DataFrame for the report
audit_df = pd.DataFrame(audit_data)

# Reorder columns for better readability
cols = ['filename', 'status', 'num_pages', 'char_count', 'valid_ratio', 'snippet']
audit_df = audit_df[cols]

# --- FINAL REPORT ---
print("\n=== PDF HEALTH AUDIT REPORT ===")
print(f"Total checked: {len(audit_df)}")
print("\nStatus Distribution:")
print(audit_df['status'].value_counts())

print("\nDetailed breakdown (Top 10):")
# We display the metrics clearly so anyone can see the proof
display(audit_df.head(10))

# Check if we have any failures
failures = audit_df[audit_df['status'].str.contains("FAILED")]
if not failures.empty:
    print(f"\nWARNING: Found {len(failures)} problematic files!")
    display(failures)
else:
    print("\nSUCCESS: All sampled files passed the integrity tests.")

Running detailed health check on sample documents...


  0%|          | 0/15 [00:00<?, ?it/s]


=== PDF HEALTH AUDIT REPORT ===
Total checked: 15

Status Distribution:
status
PASSED (Healthy)    15
Name: count, dtype: int64

Detailed breakdown (Top 10):


,filename,status,num_pages,char_count,valid_ratio,snippet
0,KS_Banská_Bystrica_35Cbi_59_2006_00_dokument.pdf,PASSED (Healthy),6,13769,0.817,Súd: Krajský súd Banská Bystrica Spisová značk...
1,KS_Bratislava_4CoZm_2_2018_00_dokument.pdf,PASSED (Healthy),10,13493,0.807,Súd: Krajský súd Bratislava Spisová značka: 4C...
2,KS_Trnava_31Cob_64_2011_00_dokument.pdf,PASSED (Healthy),4,11394,0.820,Súd: Krajský súd Trnava Spisová značka: 31Cob/...
3,KS_Trnava_21Cob_41_2011_00_dokument.pdf,PASSED (Healthy),4,12665,0.815,Súd: Krajský súd Trnava Spisová značka: 21Cob/...
4,KS_Nitra_15Cob_70_2022_00_dokument.pdf,PASSED (Healthy),8,13712,0.817,Súd: Krajský súd Nitra Spisová značka: 15Cob/7...
5,KS_Žilina_13Cob_7_2017_00_dokument.pdf,PASSED (Healthy),8,14309,0.817,Súd: Krajský súd Žilina Spisová značka: 13Cob/...
6,KS_Prešov_1Cob_20_2012_00_dokument.pdf,PASSED (Healthy),7,12424,0.814,Súd: Krajský súd Prešov Spisová značka: 1Cob/2...
7,KS_Bratislava_1Cob_117_2020_00_dokument.pdf,PASSED (Healthy),11,13295,0.807,Súd: Krajský súd Bratislava Spisová značka: 1C...
8,KS_Nitra_26Cob_39_2012_00_dokument.pdf,PASSED (Healthy),7,10826,0.816,Súd: Krajský súd Nitra Spisová značka: 26Cob/3...
9,KS_Bratislava_1Cob_298_2015_00_dokument.pdf,PASSED (Healthy),3,12323,0.806,Súd: Krajský súd Bratislava Spisová značka: 1C...



✅ SUCCESS: All sampled files passed the integrity tests.


## Text statistics

In [22]:
def extract_full_text(pdf) -> str:
    # I am extracting text page by page
    # filter out None to avoid errors if a page is completely blank
    valid_text = [p.extract_text() for p in pdf.pages if p.extract_text()]
    # join with double newlines to keep paragraphs separated
    return "\n\n".join(valid_text)

rows = []
print(f"Processing {len(df)} files...")

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = BASE_DIR / row["rel_path"]
    
    # safe initialization: start with basic info
    out = {"rel_path": row["rel_path"]}

    # IMPORTANT: Pre-fill all counting columns with 0.
    # If the PDF crashes later, I still want '0' in my table, not 'NaN' (empty).
    for k in KEY.keys():
        out[k] = 0
    out["has_odovodnenie"] = 0
    out["has_poucenie"] = 0
    out["has_vyrok"] = 0
    out["error"] = None # placeholder for errors

    try:
        with pdfplumber.open(path) as pdf:
            out["page_count"] = len(pdf.pages)

            t = extract_full_text(pdf)
            out["char_count"] = len(t)
            # simple estimation: 1 token is roughly 4 characters
            out["est_tokens"] = int(out["char_count"] / 4)

            # normalizing text for regex search
            # raw_lower = for reading, ascii_norm = for searching (removes fada/makcen)
            raw_lower, ascii_norm = norm_pair(t)

            # counting keywords from my regex bank
            for k, rx in KEY.items():
                out[k] = len(rx.findall(ascii_norm))

            # checking for document structure (headers)
            out["has_odovodnenie"] = int(bool(RE_ODOV.search(ascii_norm)))
            out["has_poucenie"] = int(bool(RE_POUC.search(ascii_norm)))
            out["has_vyrok"] = int(bool(RE_VYROK.search(ascii_norm)))

            # calculating paragraph statistics to see if text is chunk-ready
            # splitting by empty lines
            pars = [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
            par_lens = [int(len(p)/4) for p in pars] 
            
            out["n_pars"] = len(pars)
            out["par_max_tokens"] = max(par_lens) if par_lens else 0
            # getting the 90th percentile to see how long the big paragraphs are
            out["par_p90_tokens"] = int(np.percentile(par_lens, 90)) if len(par_lens) >= 2 else (par_lens[0] if par_lens else 0)

    except Exception as e:
        # if file is corrupted, set counts to 0 and save the error message
        out["page_count"] = 0
        out["char_count"] = 0
        out["est_tokens"] = 0
        out["error"] = str(e)

    rows.append(out)

df_feat = pd.DataFrame(rows)

# cleanup: remove old feature columns before merging to avoid duplicates
feature_cols = [c for c in df_feat.columns if c != "rel_path"]
df = df.drop(columns=[c for c in feature_cols if c in df.columns], errors="ignore")

# joining the new features to the main table
df = df.merge(df_feat, on="rel_path", how="left")

# checking the results
df[["filename", "page_count","char_count","est_tokens","has_penalty","has_301","error"]]

Processing 152 files...


  0%|          | 0/152 [00:00<?, ?it/s]

,filename,page_count,char_count,est_tokens,has_penalty,has_301,error
0,KS_Banská_Bystrica_35Cbi_59_2006_00_dokument.pdf,6,23517,5879,48,4,None
1,KS_Bratislava_4CoZm_2_2018_00_dokument.pdf,10,44468,11117,15,2,None
2,KS_Trnava_31Cob_64_2011_00_dokument.pdf,4,11461,2865,9,1,None
3,KS_Trnava_21Cob_41_2011_00_dokument.pdf,4,16167,4041,26,2,None
4,KS_Nitra_15Cob_70_2022_00_dokument.pdf,8,35221,8805,35,7,None
...,...,...,...,...,...,...,...
147,KS_Trenčín_8Cob_74_2011_00_dokument.pdf,3,9507,2376,34,3,None
148,KS_Nitra_26Cob_28_2017_00_dokument.pdf,4,12897,3224,11,1,None
149,KS_Prešov_5Cob_11_2022_00_dokument.pdf,6,22805,5701,22,2,None
150,KS_Bratislava_2Cob_140_2023_00_dokument.pdf,14,65371,16342,50,1,None


In [21]:
# --- FINDING INCOMPLETE DOCUMENTS ---
# Goal: I want to see only the files that are missing something important.
# A valid judgment must have 3 parts: Verdict (Vyrok), Reasoning (Odovodnenie), Instruction (Poucenie).

# 1. Create a filter for "Bad" documents
# If any of these counts is 0, the document is incomplete.
# logic: (No Verdict) OR (No Reasoning) OR (No Instruction)
bad_docs = df[
    (df["has_vyrok"] == 0) | 
    (df["has_odovodnenie"] == 0) | 
    (df["has_poucenie"] == 0)
]

print("--- DATA QUALITY REPORT ---")
print(f"Total documents: {len(df)}")
print(f"Incomplete documents found: {len(bad_docs)}")

# 2. Calculate percentage
# If this number is high (e.g., over 20%), I might need to improve my Regex or check the source.
if len(df) > 0:
    percent_bad = (len(bad_docs) / len(df)) * 100
    print(f"Percentage of incomplete files: {percent_bad:.1f}%")

# 3. Show the problematic files
# I want to see WHICH part is missing.
# 0 = Missing (BAD), 1 = Present (GOOD)
print("\n--- LIST OF PROBLEMATIC FILES ---")
cols = ["filename", "has_vyrok", "has_odovodnenie", "has_poucenie"]

if not bad_docs.empty:
    display(bad_docs[cols])
else:
    print("Great news! No incomplete documents found. All files define the structure correctly.")

--- DATA QUALITY REPORT ---
Total documents: 152
Incomplete documents found: 5
Percentage of incomplete files: 3.3%

--- LIST OF PROBLEMATIC FILES ---


,filename,has_vyrok,has_odovodnenie,has_poucenie
9,KS_Bratislava_1Cob_298_2015_00_dokument.pdf,0,1,1
20,KS_Banská_Bystrica_43Cob_20_2016_00_dokument.pdf,0,1,1
34,KS_Bratislava_3Cob_464_2012_00_dokument.pdf,0,1,1
80,KS_Žilina_14Cob_124_2012_00_dokument.pdf,0,1,1
146,NS_SR_1Obdo_7_2020_00_dokument.pdf,0,1,1


## Header / footer repetitions

In [ ]:


def page_edge_lines(pdf_path: Path, n_lines=2, max_pages=10):
    top, bottom = [], []
    with pdfplumber.open(pdf_path) as pdf:
        for pg in pdf.pages[:max_pages]:
            t = (pg.extract_text() or "").strip()
            lines = [x.strip() for x in t.splitlines() if x.strip()]
            top.extend(lines[:n_lines])
            bottom.extend(lines[-n_lines:])
    return top, bottom

top_c = Counter()
bot_c = Counter()

sample_paths = df.sample(min(30, len(df)), random_state=42)["rel_path"].tolist()
for rp in sample_paths:
    p = BASE_DIR / rp
    t, b = page_edge_lines(p, n_lines=2, max_pages=10)
    top_c.update(t); bot_c.update(b)

top_df = pd.DataFrame(top_c.most_common(30), columns=["line","count"])
bot_df = pd.DataFrame(bot_c.most_common(30), columns=["line","count"])

display(top_df.head(15))
display(bot_df.head(15))


## Visualizing the Dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of tokens (how expensive will the LLM be?)
sns.histplot(df['est_tokens'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Estimated Tokens')
axes[0].set_xlabel('Tokens')
axes[0].axvline(df['est_tokens'].mean(), color='red', linestyle='--', label='Mean')

# 2. distribution of years (temporal coverage)
if df['date'].notna().sum() > 0:
    df["year"] = df["date"].dt.year
    year_counts = df["year"].dropna().astype(int).value_counts().sort_index()
    
    sns.barplot(x=year_counts.index.astype(str), y=year_counts.values, ax=axes[1])
    axes[1].set_title("Decisions by year")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Count")

# 3. top courts (Where does data come from?)
top_courts = df['court'].value_counts().head(10)
sns.barplot(
    x=top_courts.values,
    y=top_courts.index,
    hue=top_courts.index,       
    palette='magma',
    ax=axes[2],
    legend=False             
)
axes[2].set_title('Top 10 Courts')

plt.tight_layout()
plt.show()


## Deep Dive into ONE Document

In [ ]:
# choose real document from df
sample_row = df.sample(1, random_state=42).iloc[0]
path = BASE_DIR / sample_row['rel_path']

with pdfplumber.open(path) as pdf:
    print(f"Analyzing: {sample_row['filename']}")
    print(f"Total pages: {len(pdf.pages)}")

    p1 = pdf.pages[0]
    text = p1.extract_text() or ""
    print("\n--- PAGE 1 (first 800 chars) ---")
    print(text[:800])

    # fast layout sanity (how much words and their bbox)
    words = p1.extract_words()[:20]
    print("\n--- FIRST 20 WORD BOXES ---")
    for w in words:
        print(w["text"], (w["x0"], w["top"], w["x1"], w["bottom"]))


## Contract type triggers

In [ ]:
# 1.X Contract type triggers (very light heuristic)
CONTRACT_RX = {
    "sale_purchase": re.compile(r"\bkupn\w+\s+zmluv\w+\b", re.IGNORECASE),
    "work_contract": re.compile(r"\bzmluv\w+\s+o\s+diel\w+\b", re.IGNORECASE),
    "lease": re.compile(r"\bn[aá]jomn\w+\s+zmluv\w+\b", re.IGNORECASE),
    "loan_credit": re.compile(r"\b(uver\w+|úver\w+|p[oô]žičk\w+)\b", re.IGNORECASE),
    "mandate": re.compile(r"\bmand[aá]tn\w+\s+zmluv\w+\b|\bpr[ií]kazn\w+\s+zmluv\w+\b", re.IGNORECASE),
    "agency": re.compile(r"\bsprostredkovate[lľ]sk\w+\s+zmluv\w+\b", re.IGNORECASE),
    "insurance": re.compile(r"\bpoistn\w+\s+zmluv\w+\b", re.IGNORECASE),
}

# count triggers using ascii_norm from ONE PASS
for k, rx in CONTRACT_RX.items():
    df[f"ct_{k}"] = 0

# quick: reuse ONE PASS text? (we don't store it) -> do on small sample only for EDA
sample = df.sample(min(40, len(df)), random_state=42)
hits = Counter()

for _, r in sample.iterrows():
    path = BASE_DIR / r["rel_path"]
    with pdfplumber.open(path) as pdf:
        text = "\n\n".join([(p.extract_text() or "") for p in pdf.pages])
    a = unidecode(text).lower()
    for k, rx in CONTRACT_RX.items():
        if rx.search(a):
            hits[k] += 1

pd.DataFrame(hits.most_common(), columns=["contract_type","docs_in_sample"])


## Does decisions talk about contractual penalty

In [ ]:
print(df["has_301"].gt(0).sum(), "docs mention §301")
print(df["has_penalty"].gt(0).sum(), "docs mention zmluvná pokuta")
print(df["has_oz_545a"].gt(0).sum(), "docs mention OZ §545a")
print(df["has_oz_544"].gt(0).sum(), "docs mention OZ §544")

## 1.7b Better categories

In [ ]:
# 1.7b Better categories (aligned to thesis goal: §301 moderation / reasonableness)
def label(row):
    # civil noise (you will exclude later)
    if row["has_oz_545a"] > 0 or row["has_oz_544"] > 0:
        return "CIVIL_NOISE_LIKELY"

    # main goal: discusses reasonableness/moderation in trade context
    if row["has_penalty"] > 0 and (row["has_301"] > 0) and (row["has_moder_trig"] > 0 or row["has_primer_kw"] > 0):
        return "REASONABLENESS_OR_MODERATION_LIKELY"

    # mentions penalty but no clear moderation signal
    if row["has_penalty"] > 0 and row["has_301"] > 0:
        return "PENALTY_AND_301_BUT_UNCLEAR"

    if row["has_penalty"] > 0:
        return "PENALTY_MENTION_NO_301"

    return "IRRELEVANT_LIKELY"

df["eda_label"] = df.apply(label, axis=1)
df["eda_label"].value_counts()


## Applied vs not applied

In [ ]:
# 1.7c Outcome proxy (EDA-only)
# Note: this is just a heuristic to guide manual audit sampling
RE_REDUCED = re.compile(r"\bzn[ií]žil\b|\bzni[zž]il\b|\bznížen\w*|\bznizen\w*", re.IGNORECASE)
RE_NOT_REDUCED = re.compile(r"neprist[uú]pil\w*\s+k\s+zni[zž]eniu|nebolo\s+potrebn\w*\s+.*modera|nevyhovel\w*\s+.*modera", re.IGNORECASE)

# We'll approximate using existing counts (full-text regex was counted in ONE PASS)
# If you also want the strict version, we can add context window later in the extraction pipeline.
df["outcome_proxy"] = "UNCLEAR"
df.loc[(df["has_301"]>0) & (df["has_moder_trig"]>0), "outcome_proxy"] = "DISCUSSED"
# If you counted reduced/not_reduced explicitly in KEY, map them here. Otherwise keep DISCUSSED.

df["outcome_proxy"].value_counts()


## 1.8 Structural integrity check (the "section splitter" test)

In [ ]:

print("Odôvodnenie coverage:", (df["has_odovodnenie"] == 1).mean())
print("Poučenie coverage:", (df["has_poucenie"] == 1).mean())
print("Výrok coverage:", (df["has_vyrok"] == 1).mean())

# show the worst cases ( a littbe bit of text or damaged layout)
df.sort_values(["has_odovodnenie","char_count"]).head(20)[
    ["filename","court","case_id","page_count","char_count","has_odovodnenie","has_poucenie","has_vyrok","eda_label"]
]


## Chunk readiness: paragraph and long paragraphs

In [ ]:
print(df[["n_pars","par_p90_tokens","par_max_tokens"]].describe().round(1))

# how much documents will need token fallback? (> 1000 tokens per paragraph)
fallback_rate = (df["par_max_tokens"] > 1000).mean()
print("Docs needing token-fallback (par_max_tokens>1000):", round(fallback_rate, 3))

plt.figure(figsize=(8,4))
sns.histplot(df["par_max_tokens"], bins=30)
plt.title("Max paragraph length (token-ish)")
plt.show()


## Interest vs penalty confusion

In [ ]:
# if document has percentage and interest trig, it is risky
df["interest_risk"] = (df["has_percent"] > 0) & (df["has_interest"] > 0)

print("Docs with % and interest language:", df["interest_risk"].mean())

#  MODERATION_LIKELY vs others
pd.crosstab(df["eda_label"], df["interest_risk"], normalize="index")


In [ ]:
# 1.10b Interest vs penalty (proximity-based sanity on a small sample)
WINDOW = 250  # chars
sample = df.sample(min(25, len(df)), random_state=42)

def proximity_counts(text: str):
    if not text:
        return (0,0)
    t = text
    a = unidecode(t).lower()
    pct_positions = [m.start() for m in re.finditer(r"\d+(?:[.,]\d+)?\s*%", t)]
    penalty_positions = [m.start() for m in re.finditer(r"zmluvn\w*\s+pokut\w*", a)]
    interest_positions = [m.start() for m in re.finditer(r"(urok|úrok|omeskan|omeškan)", a)]

    def near(pos_list, anchor_list):
        for p in pos_list:
            if any(abs(p - q) <= WINDOW for q in anchor_list):
                return True
        return False

    return int(near(pct_positions, penalty_positions)), int(near(pct_positions, interest_positions))

rows = []
for _, r in sample.iterrows():
    path = BASE_DIR / r["rel_path"]
    with pdfplumber.open(path) as pdf:
        text = "\n\n".join([(p.extract_text() or "") for p in pdf.pages])
    near_penalty, near_interest = proximity_counts(text)
    rows.append({"filename": r["filename"], "near_penalty_pct": near_penalty, "near_interest_pct": near_interest})

prox = pd.DataFrame(rows)
prox.mean(numeric_only=True)


## Manual audit

In [ ]:
OUT = BASE_DIR / "data" / "eda_outputs"
OUT.mkdir(parents=True, exist_ok=True)

def stratified_sample(df_, n=10):
    out = []
    for label, g in df_.groupby("eda_label"):
        out.append(g.sample(min(n, len(g)), random_state=42))
    return pd.concat(out).sample(frac=1, random_state=42)

audit = stratified_sample(df, n=8)[
    ["filename","rel_path","court","case_id","date","type","eda_label",
     "oblast_first","court_tier",
     "has_penalty","has_301","has_moder_trig","has_primer_kw","has_invalidity",
     "has_percent","has_money","has_interest",
     "page_count","char_count","est_tokens"]
].copy()

audit["MAN_is_trade_meta"] = ""   # should be yes (Obchodné právo)
audit["MAN_really_301"] = ""      # yes/no/unclear
audit["MAN_reasonableness_discussed"] = ""  # yes/no
audit["MAN_outcome"] = ""         # reduced / upheld / denied / unclear

audit.to_csv(OUT/"manual_audit_sample.csv", index=False)
audit.head()


# Meststky sud okresny sud here?

In [ ]:
df[df["court_tier"].isin(["OS","MS"])][["court","case_id","type","date","rel_path"]].head(30)
